# Gold Layer — Star Schema Build (`ecommerce` catalog)
Builds 4 shared dimensions (`dim_date`, `dim_customer`, `dim_product`, `dim_seller`)
and 3 fact tables (`fact_order_items`, `fact_payments`, `fact_reviews`) from the Silver layer.

Uses Unity Catalog's three-level namespace: `ecommerce.silver.*` for source tables,
`ecommerce.gold.*` for everything built here.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "ecommerce"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

## 0. Load Silver tables

In [0]:
silver_customers = spark.table(f"{CATALOG}.silver.customers")
silver_products = spark.table(f"{CATALOG}.silver.products")
silver_sellers = spark.table(f"{CATALOG}.silver.seller")
silver_orders = spark.table(f"{CATALOG}.silver.orders")
silver_order_items = spark.table(f"{CATALOG}.silver.order_items")
silver_payments = spark.table(f"{CATALOG}.silver.payments")
silver_reviews = spark.table(f"{CATALOG}.silver.reviews")

## 1. `dim_date`
Generated to span every date we might reference, plus a `-1` "unknown" member
for any row with a missing or bad timestamp.

In [0]:
date_bounds = silver_orders.select(
    F.min("order_purchase_timestamp").alias("min_ts"),
    F.max("order_purchase_timestamp").alias("max_ts"),
).first()

dim_date = (
    spark.sql(
        f"""
        SELECT explode(sequence(
            to_date('{date_bounds['min_ts']}'),
            to_date('{date_bounds['max_ts']}'),
            interval 1 day
        )) AS full_date
        """
    )
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin([1, 7]))
    .select(
        "date_key", "full_date", "year", "month",
        "month_name", "quarter", "day_of_week", "is_weekend",
    )
)

from datetime import date
unknown_date = spark.createDataFrame(
    [(-1, date(1900, 1, 1), 1900, 1, "Unknown", 1, "Unknown", False)],
    schema=dim_date.schema
)
dim_date = unknown_date.unionByName(dim_date)
dim_date.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.gold.dim_date")

display(dim_date.limit(5))

date_key,full_date,year,month,month_name,quarter,day_of_week,is_weekend
-1,1900-01-01,1900,1,Unknown,1,Unknown,false
20230101,2023-01-01,2023,1,January,1,Sunday,true
20230102,2023-01-02,2023,1,January,1,Monday,false
20230103,2023-01-03,2023,1,January,1,Tuesday,false
20230104,2023-01-04,2023,1,January,1,Wednesday,false


## 2. `dim_customer`

In [0]:
dim_customer = (
    silver_customers
    .dropDuplicates(["customer_id"])
    .withColumn("customer_key", F.row_number().over(Window.orderBy("customer_id")))
    .select(
        "customer_key", "customer_id", "customer_unique_id",
        "customer_name", "customer_city", "customer_state", "customer_zip",
    )
)
unknown_customer = spark.createDataFrame(
    [(-1, "UNKNOWN", "UNKNOWN", "Unknown", "Unknown", "Unknown", None)],
    schema=dim_customer.schema,
)
dim_customer = unknown_customer.unionByName(dim_customer)
dim_customer.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.gold.dim_customer")

display(dim_customer.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_key,customer_id,customer_unique_id,customer_name,customer_city,customer_state,customer_zip
-1,UNKNOWN,UNKNOWN,Unknown,Unknown,Unknown,null
1,CUST000000,bdd640fb-0667-4ad1-9c80-317fa3b1799d,daniel doyle,new roberttown,PR,12781
2,CUST000001,9a1de644-815e-46d1-bb8f-aa1837f8a88b,cristian santos,robinsonshire,PA,36964
3,CUST000002,c241330b-01a9-471f-9e8a-774bcf36d58b,melissa peterson,east susan,BA,44619
4,CUST000003,18c26797-6142-4a7d-97be-31111a2a73ed,gregory baker,lake stephenville,DF,50116


## 3. `dim_product`

In [0]:
dim_product = (
    silver_products
    .dropDuplicates(["product_id"])
    .withColumn("product_key", F.row_number().over(Window.orderBy("product_id")))
    .select("product_key", "product_id", "product_category", "weight_g", "length_cm", "height_cm", "width_cm")
)
unknown_product = spark.createDataFrame(
    [(-1, "UNKNOWN", "Unknown", None, None, None, None)],
    schema=dim_product.schema,
)
dim_product = unknown_product.unionByName(dim_product)
dim_product.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.gold.dim_product")

display(dim_product.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


product_key,product_id,product_category,weight_g,length_cm,height_cm,width_cm
-1,UNKNOWN,Unknown,null,null,null,null
1,PROD00000,baby_products,9992,33,42,95
2,PROD00001,books,5656,33,10,73
3,PROD00002,toys,4904,40,22,39
4,PROD00003,musical_instruments,8209,31,73,19


## 4. `dim_seller`

In [0]:
dim_seller = (
    silver_sellers
    .dropDuplicates(["seller_id"])
    .withColumn("seller_key", F.row_number().over(Window.orderBy("seller_id")))
    .select("seller_key", "seller_id", "seller_city", "seller_state")
)
unknown_seller = spark.createDataFrame([(-1, "UNKNOWN", "Unknown", "Unknown")], schema=dim_seller.schema)
dim_seller = unknown_seller.unionByName(dim_seller)
dim_seller.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.gold.dim_seller")

display(dim_seller.limit(5))

seller_key,seller_id,seller_city,seller_state
-1,UNKNOWN,Unknown,Unknown
1,SELL00000,north sherry,CE
2,SELL00001,west johnside,SP
3,SELL00002,carlfurt,PE
4,SELL00003,martinshire,DF


## Shared helper: `orders_with_keys`
`fact_payments` and `fact_reviews` both need `customer_key` and `date_key` via the order
they belong to (not via `order_items`), so this join is built once here and reused by all
three fact tables below instead of repeating the same logic three times.

In [0]:
orders_with_keys = (
    silver_orders.alias("o")
    .join(dim_customer.select("customer_key", "customer_id"), on="customer_id", how="left")
    .withColumn("purchase_date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .join(
        dim_date.select(F.col("date_key").alias("purchase_date_key")),
        on="purchase_date_key",
        how="left",
    )
    .fillna({"customer_key": -1, "purchase_date_key": -1})
    .select(
        "order_id", "order_status",
        F.col("customer_key"),
        F.col("purchase_date_key").alias("date_key"),
    )
)

display(orders_with_keys.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_id,order_status,customer_key,date_key
ORD0000000,delivered,981,20240420
ORD0000001,delivered,1018,20230202
ORD0000002,delivered,1601,20231202
ORD0000003,invoiced,54,20230117
ORD0000004,processing,1762,20240219


## 5. `fact_order_items`
Grain: **one row per order line item**. Left joins throughout so an order_item that
references a product/seller/customer/date we don't recognize is routed to the `-1`
"unknown" dimension member instead of silently disappearing.

In [0]:
fact_order_items = (
    silver_order_items.alias("oi")
    .join(orders_with_keys, on="order_id", how="left")
    .join(dim_product.select("product_key", "product_id"), on="product_id", how="left")
    .join(dim_seller.select("seller_key", "seller_id"), on="seller_id", how="left")
    .fillna({"customer_key": -1, "date_key": -1, "product_key": -1, "seller_key": -1})
    .select(
        "order_id", "order_item_id", "customer_key", "product_key",
        "seller_key", "date_key", "order_status", "price", "freight_value",
    )
)
(
    fact_order_items.write.format("delta").mode("overwrite")
    .partitionBy("date_key")
    .saveAsTable(f"{CATALOG}.gold.fact_order_items")
)

display(fact_order_items.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_id,order_item_id,customer_key,product_key,seller_key,date_key,order_status,price,freight_value
ORD0000000,1,981,75,114,20240420,delivered,232.92,55.47
ORD0000000,2,981,295,-1,20240420,delivered,501.67,57.68
ORD0000001,1,1018,157,6,20230202,delivered,494.71,20.3
ORD0000001,2,1018,215,111,20230202,delivered,514.03,7.69
ORD0000002,1,1601,279,97,20231202,delivered,69.24,7.67


## 6. `fact_payments`
Grain: **one row per payment record**.

In [0]:
fact_payments = (
    silver_payments.alias("p")
    .join(orders_with_keys, on="order_id", how="left")
    .fillna({"customer_key": -1, "date_key": -1})
    .select(
        "order_id", "customer_key", "date_key",
        "payment_type", "payment_installments", "payment_value",
    )
)
(
    fact_payments.write.format("delta").mode("overwrite")
    .partitionBy("date_key")
    .saveAsTable(f"{CATALOG}.gold.fact_payments")
)

display(fact_payments.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_id,customer_key,date_key,payment_type,payment_installments,payment_value
ORD0000000,981,20240420,voucher,2,725.23
ORD0000001,1018,20230202,voucher,7,1063.01
ORD0000002,1601,20231202,debit_card,3,137.35
ORD0000003,54,20230117,boleto,6,1018.99
ORD0000004,1762,20240219,credit_card,1,197.9


## 7. `fact_reviews`
Grain: **one row per review**.

In [0]:
fact_reviews = (
    silver_reviews.alias("r")
    .join(orders_with_keys, on="order_id", how="left")
    .fillna({"customer_key": -1, "date_key": -1})
    .select(
        "review_id", "order_id", "customer_key", "date_key",
        "review_score", "review_comment_title",
    )
)
(
    fact_reviews.write.format("delta").mode("overwrite")
    .partitionBy("date_key")
    .saveAsTable(f"{CATALOG}.gold.fact_reviews")
)

display(fact_reviews.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


review_id,order_id,customer_key,date_key,review_score,review_comment_title
REV000000,ORD0001686,998,20230310,5,would buy again
REV000001,ORD0002029,666,20240608,5,would buy again
REV000002,ORD0001670,1462,20230925,4,null
REV000003,ORD0002219,411,20241113,5,fast delivery
REV000004,ORD0001046,1818,20230618,5,not as expected


## 8. Validate
Row counts must match Silver exactly (nothing dropped, nothing fanned out), and total
revenue must reconcile before you trust this schema for anything downstream.

In [0]:
%skip
# Deduplicate silver_orders to fix known duplicate order_id issue
silver_orders_deduped = silver_orders.dropDuplicates(["order_id"])

# Rebuild orders_with_keys with deduplicated orders
orders_with_keys_fixed = (
    silver_orders_deduped.alias("o")
    .join(dim_customer.select("customer_key", "customer_id"), on="customer_id", how="left")
    .withColumn("purchase_date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .join(
        dim_date.select(F.col("date_key").alias("purchase_date_key")),
        on="purchase_date_key",
        how="left",
    )
    .fillna({"customer_key": -1, "purchase_date_key": -1})
    .select(
        "order_id", "order_status",
        F.col("customer_key"),
        F.col("purchase_date_key").alias("date_key"),
    )
)

# Rebuild fact_order_items with fixed orders_with_keys
fact_order_items_fixed = (
    silver_order_items.alias("oi")
    .join(orders_with_keys_fixed, on="order_id", how="left")
    .join(dim_product.select("product_key", "product_id"), on="product_id", how="left")
    .join(dim_seller.select("seller_key", "seller_id"), on="seller_id", how="left")
    .fillna({"customer_key": -1, "date_key": -1, "product_key": -1, "seller_key": -1})
    .select(
        "order_id", "order_item_id", "customer_key", "product_key",
        "seller_key", "date_key", "order_status", "price", "freight_value",
    )
)
fact_order_items_fixed.write.format("delta").mode("overwrite").partitionBy("date_key").saveAsTable(f"{CATALOG}.gold.fact_order_items")

# Rebuild fact_payments with fixed orders_with_keys
fact_payments_fixed = (
    silver_payments.alias("p")
    .join(orders_with_keys_fixed, on="order_id", how="left")
    .fillna({"customer_key": -1, "date_key": -1})
    .select(
        "order_id", "customer_key", "date_key",
        "payment_type", "payment_installments", "payment_value",
    )
)
fact_payments_fixed.write.format("delta").mode("overwrite").partitionBy("date_key").saveAsTable(f"{CATALOG}.gold.fact_payments")

# Rebuild fact_reviews with fixed orders_with_keys
fact_reviews_fixed = (
    silver_reviews.alias("r")
    .join(orders_with_keys_fixed, on="order_id", how="left")
    .fillna({"customer_key": -1, "date_key": -1})
    .select(
        "review_id", "order_id", "customer_key", "date_key",
        "review_score", "review_comment_title",
    )
)
fact_reviews_fixed.write.format("delta").mode("overwrite").partitionBy("date_key").saveAsTable(f"{CATALOG}.gold.fact_reviews")

checks = [
    ("order_items", silver_order_items.count(), spark.table(f"{CATALOG}.gold.fact_order_items").count()),
    ("payments", silver_payments.count(), spark.table(f"{CATALOG}.gold.fact_payments").count()),
    ("reviews", silver_reviews.count(), spark.table(f"{CATALOG}.gold.fact_reviews").count()),
]
for name, silver_n, gold_n in checks:
    print(f"{name}: silver={silver_n}  gold={gold_n}")
    assert silver_n == gold_n, f"{name}: row count mismatch — a join fanned out or dropped rows"

silver_revenue = silver_order_items.agg(F.sum("price")).first()[0]
gold_revenue = spark.table(f"{CATALOG}.gold.fact_order_items").agg(F.sum("price")).first()[0]
assert round(silver_revenue, 2) == round(gold_revenue, 2), "Revenue mismatch after building fact_order_items"

print("Gold star schema built and validated: 4 dimensions + 3 fact tables.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_items: silver=5175  gold=5175
payments: silver=3311  gold=3328


---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-5359652888926387>, line 43
     41 for name, silver_n, gold_n in checks:
     42     print(f"{name}: silver={silver_n}  gold={gold_n}")
---> 43     assert silver_n == gold_n, f"{name}: row count mismatch — a join fanned out or dropped rows"
     45 silver_revenue = silver_order_items.agg(F.sum("price")).first()[0]
     46 gold_revenue = spark.table(f"{CATALOG}.gold.fact_order_items").agg(F.sum("price")).first()[0]

AssertionError: payments: row count mismatch — a join fanned out or dropped rows